In [ ]:
from transformers import AutoModelForCTC, AutoProcessor, AutoTokenizer
import librosa
import torch

In [ ]:
model = AutoModelForCTC.from_pretrained("Korla/Wav2Vec2BertForCTC-hsb-0").to("cpu").eval().to(torch.float16)
processor = AutoProcessor.from_pretrained("Korla/Wav2Vec2BertForCTC-hsb-0")

In [ ]:
from torch.export.dynamic_shapes import Dim
dummy = torch.randn(1,16000, dtype=torch.float32)
inputs = processor(dummy, sampling_rate=16000, return_tensors="pt").to(torch.float16)

dynamic_shapes = {
    "input": {0: 1, 1: Dim.AUTO, 2: 160},
    "output": {0: 1, 1: Dim.AUTO, 2: 41}, # vocab size
}

torch.onnx.export(model, inputs["input_features"], "wav2vec2.onnx", input_names=["input"], output_names=["output"], dynamic_axes=dynamic_shapes, dynamo=True)

In [4]:
import onnxruntime


ort_session = onnxruntime.InferenceSession(
    "./wav2vec2.onnx", providers=["CPUExecutionProvider"]
)

In [6]:
waveform, sample_rate = librosa.load("1.wav", sr=16000)
wav = waveform[:16000*10] # trim to 10 seconds for testing
inputs = processor(wav, sampling_rate=16000, return_tensors="pt").to(torch.float16)
onnx_inputs = inputs["input_features"].numpy()

In [ ]:
warm_up = 3
samples = 10
times = []

for i in range(warm_up):
    onnxruntime_outputs = ort_session.run(None, {"input": onnx_inputs})[0][0][0]
    torch_outputs = model(inputs["input_features"]).logits.detach()
for i in range(samples):
    t1 = time.time()
    onnxruntime_outputs = ort_session.run(None, {"input": onnx_inputs})[0][0][0]
    t2 = time.time()
    torch_outputs = model(inputs["input_features"]).logits.detach()
    t3 = time.time()
    times.append((t2 - t1, t3 - t2))
print(f"Average ONNX Runtime inference time: {sum(t[0] for t in times) / samples:.4f} seconds")
print(f"Average PyTorch inference time: {sum(t[1] for t in times) / samples:.4f} seconds")